1. 字符串输出解析器StrOutputParser

In [7]:
# 1. 获取大模型
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import os
import dotenv

# 加载配置文件
dotenv.load_dotenv()

# 1. 提供大模型
chat_model = ChatOpenAI(
    model_name="gpt-4o-mini"
)

# 2. 调用大模型
response = chat_model.invoke('什么是大语言模型？')

print(type(response)) # AiMessage

# 3. 如何获取一个字符串的输出结果
# 方式1：自已调用输出结果的content属性
# print(response.content)

# 方式2：使用StrOutputParser
parser = StrOutputParser()
str_response = parser.invoke(response)
print(str_response)
print(type(str_response))

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


<class 'langchain_core.messages.ai.AIMessage'>
大语言模型（Large Language Model，LLM）是一种基于深度学习的自然语言处理模型，能够理解、生成和处理人类语言。它们通常是通过在大规模文本数据上进行训练得来的，并且具有大量的参数，使其能够捕捉语言中的复杂结构和模式。

大语言模型的特点包括：

1. **规模庞大**：包括数亿到数千亿个参数，允许模型学习更丰富的语言信息。
2. **预训练和微调**：通常经历预训练阶段（在大量文本上学习语言的基本结构）和微调阶段（在特定任务或领域上进行调整）。
3. **上下文理解**：能够理解和生成长文本，具备较强的上下文理解能力。
4. **多样性**：能够完成多种自然语言处理任务，如问答、翻译、文本生成、情感分析等。

大语言模型的应用广泛，包括聊天机器人、智能客服、内容生成、自动翻译等。近年来，随着计算能力的提升和大数据的可用性，这类模型的发展迅速，并在各个领域取得了显著的成果。
<class 'str'>


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


2. JSON解析器JsonOutputParser

- 方式1：

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model_name="gpt-4o-mini")

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个靠谱的{role}"),
    ("human", "{question}"),
])

prompt = chat_prompt_template.invoke(input={
    "role": "人工智能的专家", 
    "question": "人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式的数据"
})

response = chat_model.invoke(prompt)

parser = JsonOutputParser()
json_response = parser.invoke(response)
print(response)
print(json_response)
print(type(json_response))

方式2：

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model_name="gpt-4o-mini")

joke_query = "讲一个笑话"

# 定义json解析器
parser = JsonOutputParser()

prompt_template = PromptTemplate.from_template(
    template="回答用户的查询\n 满足的格式为{format_instructions}\n 问题为{question}",
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

prompt = prompt_template.invoke(input={"question": joke_query})
response = chat_model.invoke(prompt)

json_result = parser.invoke(response)
print(json_result)

拓展：链式调用 

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model_name="gpt-4o-mini")

joke_query = "讲一个笑话"

# 定义json解析器
parser = JsonOutputParser()

prompt_template = PromptTemplate.from_template(
    template="回答用户的查询\n 满足的格式为{format_instructions}\n 问题为{question}",
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 写法1 
# prompt = prompt_template.invoke(input={"question": joke_query})
# response = chat_model.invoke(prompt)
# json_result = parser.invoke(response)

# 写法2 
chain = prompt_template | chat_model | parser

chain.invoke(input={"question": joke_query})

print(json_result)

3. XMLOutputParser XML输出解析器的使用

- 举例1：自己在提示词模板中写明使用XML格式

In [ ]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

actor_query = "周星驰的简短电影记录"
response = chat_model.invoke(f"请生成{actor_query}, 将影片附在<movie></movie>标签中")

print(type(response))
print(response.content)

- 举例2

In [ ]:
from langchain_core.output_parsers import XMLOutputParser

parser = XMLOutputParser()


chat_model = ChatOpenAI(model="gpt-4o-mini")

actor_query = "生成汤姆.汉克斯的简短电影记录，请使用中文回复"

# 生成提示词模板
prompt_template1 = PromptTemplate.from_template(
    template="用户的问题：{query}\n使用的格式：{format_instructions}"
)

prompt_template2 = prompt_template1.partial(
    format_instructions=parser.get_format_instructions()
)

chain = prompt_template2 | chat_model
response = chain.invoke({"query": actor_query})

print(response.content)